In [6]:
from typing import List
from collections import Counter

class Solution:
    def findXSum(self, nums: List[int], k: int, x: int) -> List[int]:

        n = len(nums)
        ans = []
        
        for i in range(n - k + 1):
            sub = nums[i:i+k]
            freq = Counter(sub)
            
            # 依據 (-freq, -value) 排序
            sorted_items = sorted(freq.items(), key=lambda item: (-item[1], -item[0]))
            
            total = 0
            count = 0
            for val, f in sorted_items:
                if count == x:
                    break
                total += val * f
                count += 1
            
            ans.append(total)
        
        return ans


In [7]:
def findXSum(nums: List[int], k: int, x: int) -> List[int]:
    n = len(nums)
    ans = []
    
    for i in range(n - k + 1):
        sub = nums[i:i+k]
        freq = Counter(sub)
        
        # 依據 (-freq, -value) 排序
        sorted_items = sorted(freq.items(), key=lambda item: (-item[1], -item[0]))
        
        total = 0
        count = 0
        for val, f in sorted_items:
            if count == x:
                break
            total += val * f
            count += 1
        
        ans.append(total)
    
    return ans


In [8]:
nums = [1,1,2,2,3,4,2,3]
k = 6
x = 2

findXSum(nums, k, x)

[6, 10, 12]

---
### **Use Heap**

In [9]:
import heapq
from collections import Counter, defaultdict

class Solution:
    def findXSum(self, nums, k, x):
        n = len(nums)
        if n == 0: 
            return []
        freq = Counter(nums[:k])

        # 比較規則：當 (f1, v1) 與 (f2, v2) 比較時，
        # 我們認為 (f, v) 大的意義是 f 大；若 f 相等則 v 大。
        # top_heap 保存目前被選為 top 的元素，用 min-heap 表示「top 中最弱」 (f small, v small)
        # rest_heap 保存非 top 中的元素，用 max-heap 表示「rest 中最強」(-f, -v)
        top_heap = []   # elements: (f, v) ; min-heap by (f, v)
        rest_heap = []  # elements: (-f, -v) ; pop -> strongest in rest

        in_top = defaultdict(bool)  # membership map: value -> bool (是否在 top)
        # 初始化：根據初始 freq 決定 top x
        items = list(freq.items())  # (val, f)
        # sort by (freq desc, val desc)
        items.sort(key=lambda iv: (-iv[1], -iv[0]))
        x_selected = items[:x]  # top x items (may be fewer if distinct < x)

        xsum = 0
        top_size = 0
        for v, f in x_selected:
            in_top[v] = True
            heapq.heappush(top_heap, (f, v))
            xsum += f * v
            top_size += 1

        # rest: everything else
        for v, f in items[x:]:
            in_top[v] = False
            heapq.heappush(rest_heap, (-f, -v))

        # helper: clean & peek/pop valid entries
        def clean_top_peek():
            # return (f,v) for weakest top element, or None
            while top_heap:
                f, v = top_heap[0]
                if freq.get(v, 0) != f or not in_top.get(v, False):
                    heapq.heappop(top_heap)  # stale entry
                    continue
                return (f, v)
            return None

        def clean_rest_peek():
            # return (f,v) for strongest rest element, or None
            while rest_heap:
                nf, nv = rest_heap[0]
                f, v = -nf, -nv
                if freq.get(v, 0) != f or in_top.get(v, False):
                    heapq.heappop(rest_heap)  # stale entry
                    continue
                return (f, v)
            return None

        def pop_top_valid():
            # pop and return a valid weakest top (f, v) or None
            while top_heap:
                f, v = heapq.heappop(top_heap)
                if freq.get(v, 0) == f and in_top.get(v, False):
                    return (f, v)
            return None

        def pop_rest_valid():
            # pop and return a valid strongest rest (f, v) or None
            while rest_heap:
                nf, nv = heapq.heappop(rest_heap)
                f, v = -nf, -nv
                if freq.get(v, 0) == f and not in_top.get(v, False):
                    return (f, v)
            return None

        # initial answer for first window
        ans = [xsum]

        # slide window
        for i in range(k, n):
            out_num = nums[i - k]
            in_num = nums[i]

            # --- process out_num (decrement frequency) ---
            old = freq.get(out_num, 0)
            new = old - 1
            if new <= 0:
                if old > 0:
                    # existed before, now removed
                    freq.pop(out_num, None)
                else:
                    # shouldn't happen
                    pass
            else:
                freq[out_num] = new

            # if out_num currently in top, adjust xsum and possibly membership
            if in_top.get(out_num, False):
                # xsum changes by (new - old) * out_num
                xsum += (new - old) * out_num
                if new == 0:
                    # element is gone -> leave top
                    in_top[out_num] = False
                    top_size -= 1
                    # push into rest? no, freq==0 -> no need
                else:
                    # push updated entry into top heap (lazy)
                    heapq.heappush(top_heap, (new, out_num))
            else:
                # out_num was in rest (or not present). If new>0 push updated into rest heap
                if new > 0:
                    heapq.heappush(rest_heap, (-new, -out_num))

            # --- process in_num (increment frequency) ---
            old = freq.get(in_num, 0)
            new = old + 1
            freq[in_num] = new

            if in_top.get(in_num, False):
                # already in top: adjust xsum and push updated entry
                xsum += (new - old) * in_num
                heapq.heappush(top_heap, (new, in_num))
            else:
                # was in rest or new element: push updated into rest heap
                heapq.heappush(rest_heap, (-new, -in_num))

            # --- re-balance to ensure exactly top_size == x (or ≤ distinct) and top contains top x by order ---
            # 1) fill top if top_size < x
            while top_size < x:
                candidate = pop_rest_valid()
                if candidate is None:
                    break
                f, v = candidate
                in_top[v] = True
                top_size += 1
                xsum += f * v
                heapq.heappush(top_heap, (f, v))

            # 2) if top_size > x (happens if some element's freq went 0 while in_top was True)
            while top_size > x:
                candidate = pop_top_valid()
                if candidate is None:
                    break
                f, v = candidate
                # move it to rest (only if still positive freq)
                in_top[v] = False
                top_size -= 1
                xsum -= f * v
                if freq.get(v, 0) > 0:
                    heapq.heappush(rest_heap, (-freq[v], -v))

            # 3) now fix cross-over: if strongest in rest > weakest in top => swap
            while True:
                weakest_top = clean_top_peek()  # (f_w, v_w)
                strongest_rest = clean_rest_peek()  # (f_r, v_r)
                if weakest_top is None or strongest_rest is None:
                    break
                f_w, v_w = weakest_top
                f_r, v_r = strongest_rest
                # if rest's strongest better than top's weakest -> swap
                # "better" means (f_r > f_w) or (f_r == f_w and v_r > v_w)
                if (f_r > f_w) or (f_r == f_w and v_r > v_w):
                    # pop valid ones
                    pop_top_valid()     # removes weakest top (f_w, v_w)
                    pop_rest_valid()    # removes strongest rest (f_r, v_r)

                    # update membership and heaps and xsum
                    in_top[v_w] = False
                    in_top[v_r] = True

                    top_size -= 1
                    top_size += 1

                    xsum -= f_w * v_w
                    xsum += f_r * v_r

                    # push updated entries to heaps (with current freq)
                    if freq.get(v_w, 0) > 0:
                        heapq.heappush(rest_heap, (-freq[v_w], -v_w))
                    heapq.heappush(top_heap, (freq[v_r], v_r))

                    # continue loop to check if more swaps needed
                else:
                    break

            ans.append(xsum)

        return ans


In [10]:
nums = [3,8,7,8,7,5]
k = 2
x = 2

Solution().findXSum(nums, k, x)

[11, 15, 15, 15, 12]